In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
train_path = 'Groceries data train.csv'
test_path  = 'Groceries data test.csv'
train_raw  = pd.read_csv(train_path)
test_raw   = pd.read_csv(test_path)

In [3]:
# Get unique users and items
unique_users = train_raw['User_id'].unique()
unique_items = train_raw['itemDescription'].unique()
print("Number of unique users:", len(unique_users))
print("Number of unique items:", len(unique_items))

Number of unique users: 3493
Number of unique items: 167


In [4]:
user_item_matrix = train_raw.pivot_table(
    index='User_id',
    columns='itemDescription',
    values='Date',
    aggfunc='count',
    fill_value=0
)
user_item_matrix

itemDescription,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
User_id,,,,,,,,,,,,,,,,,,,,,
1000,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,2,0,0
1002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1003,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4993,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [5]:
def similarity(user1, user2):
    dot_product = np.dot(user1, user2)
    
    magnitude_user1 = np.linalg.norm(user1)
    magnitude_user2 = np.linalg.norm(user2)

    similarity = dot_product / (magnitude_user1 * magnitude_user2)
    
    return similarity

In [6]:
def get_item_based_recommendations(user_id, n_recommendations=5):

    n_items = len(user_item_matrix.columns)
    item_similarity = np.zeros((n_items, n_items))
    for i in range(n_items):
        for j in range(n_items):
            item_similarity[i,j] = similarity(user_item_matrix.T.iloc[i], user_item_matrix.T.iloc[j])
    
    user_idx = user_item_matrix.index.get_loc(user_id)
    user_purchases = user_item_matrix.iloc[user_idx].values
    
    weighted_sums = item_similarity.dot(user_purchases)
    
    weighted_sums[user_purchases.nonzero()] = -1
    
    top_items_idx = weighted_sums.argsort()[-n_recommendations:][::-1]
    recommendations = user_item_matrix.columns[top_items_idx]
    
    return recommendations

In [7]:
def get_user_based_recommendations(user_id, n_recommendations=5):
    n_users = len(user_item_matrix.index)
    user_similarity = np.zeros((n_users, n_users))
    for i in range(n_users):
        for j in range(n_users):
            user_similarity[i,j] = similarity(user_item_matrix.iloc[i], user_item_matrix.iloc[j])
    
    user_idx = user_item_matrix.index.get_loc(user_id)
    
    similar_users = user_similarity[user_idx].argsort()[::-1][1:6]
    
    similar_user_purchases = user_item_matrix.iloc[similar_users].mean(axis=0)
    
    user_purchases = user_item_matrix.iloc[user_idx]
    similar_user_purchases[user_purchases > 0] = -1
    
    recommendations = similar_user_purchases.nlargest(n_recommendations).index
    
    return recommendations

In [ ]:
user_id = 2351

item_based_recommendations = get_item_based_recommendations(user_id)
user_based_recommendations = get_user_based_recommendations(user_id)

print("Item-based recomm  endations:", item_based_recommendations)
print("User-based recommendations:", user_based_recommendations)

Item-based recommendations: Index(['whole milk', 'other vegetables', 'soda', 'yogurt', 'rolls/buns'], dtype='object', name='itemDescription')
User-based recommendations: Index(['frozen vegetables', 'other vegetables', 'root vegetables',
       'citrus fruit', 'curd'],
      dtype='object', name='itemDescription')
